# Lenta OCR Baseline (Kaggle Notebook)

2-stage baseline for price-tag candidate extraction:
1. **Optional Threshold Calibration** (only when GT CSV is provided)
2. **Inference** (always)

Pipeline:
`video -> frames -> OCR/contours candidates -> crops -> debug CSV -> metrics -> visualizations -> zip`


In [ ]:
#Для загрузки датасета из гугла
!pip install -q gdown
!gdown --folder "https://drive.google.com/drive/folders/1XRrRB7y66RU4lxZiH7a6H_b8fOvKgOQl" -O /kaggle/working

## 1. Imports and configuration


In [ ]:
from __future__ import annotations

import importlib
import itertools
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# -----------------------------
# User-editable configuration
# -----------------------------
# Modes:
# - RUN_MODE = "single": process only INPUT_VIDEO_PATH
# - RUN_MODE = "batch": auto-discover all *.mp4 under DATA_ROOT
RUN_MODE = "batch"

DATA_ROOT = "/kaggle/working"
CASE_FILTER = None  # e.g. "43_15" to run only matching paths in batch mode
INCLUDE_UNLABELED = True

# Single mode paths (used when RUN_MODE="single")
INPUT_VIDEO_PATH = f"{DATA_ROOT}/43_15/43_15.mp4"
GT_ANNOTATIONS_PATH = f"{DATA_ROOT}/43_15/43_15.csv"  # or None

# Output
OUTPUT_DIR = "/kaggle/working/baseline_ocr_candidates_single"
BATCH_OUTPUT_ROOT = "/kaggle/working/baseline_ocr_candidates_batch"

SAMPLE_FPS = 2.0
MAX_FRAMES = None
PADDING = 0.15
RANDOM_SEED = 42

# Optional threshold calibration (CPU-heavy).
ENABLE_CALIBRATION = False

# Robot videos are sideways: enforce fixed 90 degree counterclockwise rotation.
# "none" | "cw" | "ccw" | "180"
FRAME_ROTATION_MODE = "ccw"

# Save preprocessed frame snapshots for debug visualization
SAVE_PREPROCESSED_FRAMES = True
PREPROCESS_PREVIEW_N = 8

# Optional visualization controls
TOP_N_DEBUG_FRAMES = 6
TOP_N_CROPS = 12

# OCR backend controls
# "easyocr_first" | "easyocr_only" | "tesseract_only"
OCR_BACKEND = "easyocr_first"
AUTO_INSTALL_EASYOCR = False
EASYOCR_USE_GPU = True
EASYOCR_LANG_LIST = ["ru", "en"]
EASYOCR_VERBOSE = False

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

DEFAULT_HEURISTIC_PARAMS = {
    "padding": PADDING,
    "seed_score_thr": 0.7,
    "neighbor_radius_scale": 2.0,
    "nms_iou": 0.5,
    "min_area_ratio": 0.0005,
    "max_area_ratio": 0.25,
    "aspect_ratio_min": 0.2,
    "aspect_ratio_max": 8.0,
}

plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["axes.grid"] = False

# -----------------------------
# OCR availability checks
# -----------------------------
PYTESSERACT_AVAILABLE = False
TESSERACT_BINARY_AVAILABLE = False
TESSERACT_ERROR: str | None = None

try:
    import pytesseract
    from pytesseract import Output as TesseractOutput

    PYTESSERACT_AVAILABLE = True
except Exception as exc:
    TESSERACT_ERROR = f"pytesseract import failed: {exc}"

TESSERACT_BINARY = shutil.which("tesseract")
TESSERACT_BINARY_AVAILABLE = TESSERACT_BINARY is not None
TESSERACT_AVAILABLE = PYTESSERACT_AVAILABLE and TESSERACT_BINARY_AVAILABLE

EASYOCR_AVAILABLE = False
EASYOCR_GPU_AVAILABLE = False
EASYOCR_IMPORT_ERROR: str | None = None
easyocr = None
torch = None


def _try_import_easyocr() -> tuple[bool, str | None]:
    global easyocr, torch
    try:
        easyocr = importlib.import_module("easyocr")
        torch = importlib.import_module("torch")
        return True, None
    except Exception as exc:
        return False, str(exc)


def _pip_install(cmd: list[str]) -> None:
    print("[INFO]", " ".join(cmd))
    subprocess.run(cmd, check=False)


def _import_easyocr_stack(auto_install: bool = False) -> None:
    global EASYOCR_AVAILABLE, EASYOCR_IMPORT_ERROR, EASYOCR_GPU_AVAILABLE, easyocr, torch

    ok, err = _try_import_easyocr()
    if not ok and auto_install:
        # Keep installation minimal to reduce resolver churn.
        _pip_install([sys.executable, "-m", "pip", "install", "-q", "easyocr"])
        ok, err = _try_import_easyocr()

    if not ok:
        EASYOCR_IMPORT_ERROR = err
        return

    EASYOCR_AVAILABLE = True
    EASYOCR_IMPORT_ERROR = None

    try:
        EASYOCR_GPU_AVAILABLE = bool(torch.cuda.is_available())
    except Exception:
        EASYOCR_GPU_AVAILABLE = False


_import_easyocr_stack(auto_install=AUTO_INSTALL_EASYOCR)

OCR_AVAILABLE = (
    (OCR_BACKEND in {"easyocr_first", "easyocr_only"} and EASYOCR_AVAILABLE)
    or TESSERACT_AVAILABLE
)

print(f"OCR_BACKEND={OCR_BACKEND}")
print(f"EASYOCR_AVAILABLE={EASYOCR_AVAILABLE}")
print(f"EASYOCR_GPU_AVAILABLE={EASYOCR_GPU_AVAILABLE}")
print(f"EASYOCR_USE_GPU(requested)={EASYOCR_USE_GPU}")
if EASYOCR_IMPORT_ERROR:
    print(f"[WARN] EasyOCR import/install issue: {EASYOCR_IMPORT_ERROR}")

print(f"TESSERACT_AVAILABLE={TESSERACT_AVAILABLE}")
if TESSERACT_BINARY:
    print(f"TESSERACT_BINARY={TESSERACT_BINARY}")
if TESSERACT_ERROR:
    print(f"[WARN] {TESSERACT_ERROR}")

if not OCR_AVAILABLE:
    print("[WARN] No OCR backend is currently available. Contour fallback will be used.")


## 2. Utility dataclasses


In [ ]:
@dataclass
class OCRBox:
    text: str
    confidence: float
    bbox: tuple[int, int, int, int]  # x_min, y_min, x_max, y_max


@dataclass
class Candidate:
    bbox: tuple[int, int, int, int]
    score: float
    source: str
    matched_texts: list[str]


## 3. Video inspection


In [ ]:
def inspect_video(video_path: str) -> dict[str, float | int | str]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()

    if fps <= 0:
        fps = 25.0

    duration_sec = frame_count / fps if fps > 0 else 0.0
    return {
        "video_path": video_path,
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_sec": duration_sec,
    }


## 4. Frame extraction


In [ ]:
def extract_frames(
    video_path: str,
    frames_dir: str | Path,
    sample_fps: float = 2.0,
    max_frames: int | None = None,
) -> list[dict[str, Any]]:
    frames_dir = Path(frames_dir)
    frames_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if fps <= 0:
        fps = 25.0

    if sample_fps <= 0:
        sample_fps = 1.0
    frame_step = max(1, int(round(fps / sample_fps)))

    records: list[dict[str, Any]] = []
    frame_index = 0
    saved = 0

    pbar_total = frame_count if frame_count > 0 else None
    with tqdm(total=pbar_total, desc="Extracting frames") as pbar:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            if frame_index % frame_step == 0:
                timestamp_ms = int(round((frame_index / fps) * 1000.0))
                frame_name = f"frame_{frame_index:08d}_{timestamp_ms:010d}ms.jpg"
                frame_path = frames_dir / frame_name
                cv2.imwrite(str(frame_path), frame)

                records.append(
                    {
                        "frame_index": frame_index,
                        "timestamp_ms": timestamp_ms,
                        "frame_path": str(frame_path),
                    }
                )
                saved += 1
                if max_frames is not None and saved >= max_frames:
                    break

            frame_index += 1
            if pbar_total is not None:
                pbar.update(1)

    cap.release()
    return records


## 5. Image quality functions


In [ ]:
def compute_sharpness(image: np.ndarray) -> float:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def compute_brightness(image: np.ndarray) -> float:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return float(np.mean(gray))


def compute_contrast(image: np.ndarray) -> float:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return float(np.std(gray))


def preprocess_frame_bundle(image_bgr: np.ndarray) -> dict[str, np.ndarray]:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    # Local contrast normalization is useful for shelf reflections.
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(gray)
    denoised = cv2.bilateralFilter(clahe, 7, 55, 55)
    blur = cv2.GaussianBlur(denoised, (0, 0), 1.2)
    sharpen = cv2.addWeighted(denoised, 1.55, blur, -0.55, 0)

    return {
        "gray": gray,
        "clahe": clahe,
        "denoised": denoised,
        "sharpen": sharpen,
        "ocr_ready_bgr": cv2.cvtColor(sharpen, cv2.COLOR_GRAY2BGR),
    }


def compute_frame_scores(image_bgr: np.ndarray) -> dict[str, float]:
    sharpness = compute_sharpness(image_bgr)
    brightness = compute_brightness(image_bgr)
    contrast = compute_contrast(image_bgr)

    sharpness_norm = float(np.clip(sharpness / 500.0, 0.0, 1.0))
    contrast_norm = float(np.clip(contrast / 80.0, 0.0, 1.0))
    brightness_norm = float(np.clip(1.0 - abs(brightness - 127.0) / 127.0, 0.0, 1.0))

    custom_frame_score = 0.6 * sharpness_norm + 0.2 * contrast_norm + 0.2 * brightness_norm

    return {
        "sharpness": float(sharpness),
        "brightness": float(brightness),
        "contrast": float(contrast),
        "sharpness_score": float(sharpness_norm),
        "custom_frame_score": float(custom_frame_score),
    }


def compute_crop_quality(crop: np.ndarray, candidate_score: float) -> dict[str, float]:
    if crop is None or crop.size == 0:
        return {
            "sharpness": 0.0,
            "brightness": 0.0,
            "contrast": 0.0,
            "quality_score": 0.0,
        }

    sharpness = compute_sharpness(crop)
    brightness = compute_brightness(crop)
    contrast = compute_contrast(crop)

    sharpness_norm = float(np.clip(sharpness / 500.0, 0.0, 1.0))
    contrast_norm = float(np.clip(contrast / 80.0, 0.0, 1.0))
    brightness_norm = float(np.clip(1.0 - abs(brightness - 127.0) / 127.0, 0.0, 1.0))

    quality_score = (
        0.5 * sharpness_norm
        + 0.2 * contrast_norm
        + 0.2 * brightness_norm
        + 0.1 * float(np.clip(candidate_score, 0.0, 2.0) / 2.0)
    )

    return {
        "sharpness": sharpness,
        "brightness": brightness,
        "contrast": contrast,
        "quality_score": float(quality_score),
    }


## 6. OCR adapter


In [ ]:
_ocr_warning_printed = False
_easyocr_warning_printed = False
_easyocr_reader = None
OCR_BACKEND_IN_USE = "none"


def _safe_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except Exception:
        return default


def _run_tesseract_ocr(image: np.ndarray) -> list[OCRBox]:
    global _ocr_warning_printed

    if not TESSERACT_AVAILABLE:
        if not _ocr_warning_printed:
            warnings.warn(
                "pytesseract or tesseract binary is unavailable. Returning empty OCR list.",
                RuntimeWarning,
            )
            _ocr_warning_printed = True
        return []

    lang_order = ["rus+eng", "eng"]
    last_exc: Exception | None = None

    for lang in lang_order:
        try:
            data = pytesseract.image_to_data(
                image,
                lang=lang,
                output_type=TesseractOutput.DICT,
                config="--psm 6",
            )
        except Exception as exc:
            last_exc = exc
            continue

        boxes: list[OCRBox] = []
        n = len(data.get("text", []))
        for i in range(n):
            text = str(data.get("text", [""])[i]).strip()
            if not text:
                continue

            conf_raw = data.get("conf", ["0"])[i]
            conf = _safe_float(conf_raw, default=-1.0)
            if conf < 0:
                conf = 0.0

            left = int(_safe_float(data.get("left", [0])[i], 0.0))
            top = int(_safe_float(data.get("top", [0])[i], 0.0))
            width = int(_safe_float(data.get("width", [0])[i], 0.0))
            height = int(_safe_float(data.get("height", [0])[i], 0.0))
            if width <= 0 or height <= 0:
                continue

            x_min = max(0, left)
            y_min = max(0, top)
            x_max = max(x_min + 1, left + width)
            y_max = max(y_min + 1, top + height)

            boxes.append(
                OCRBox(
                    text=text,
                    confidence=float(conf),
                    bbox=(x_min, y_min, x_max, y_max),
                )
            )
        return boxes

    if not _ocr_warning_printed and last_exc is not None:
        warnings.warn(
            f"Tesseract OCR call failed ({last_exc}). Returning empty OCR list.",
            RuntimeWarning,
        )
        _ocr_warning_printed = True

    return []


def _init_easyocr_reader() -> Any | None:
    global _easyocr_reader, OCR_BACKEND_IN_USE, _easyocr_warning_printed

    if _easyocr_reader is not None:
        return _easyocr_reader

    if not EASYOCR_AVAILABLE:
        if not _easyocr_warning_printed:
            warnings.warn(
                "EasyOCR is unavailable. Falling back to Tesseract/contours.",
                RuntimeWarning,
            )
            _easyocr_warning_printed = True
        return None

    requested_gpu = bool(EASYOCR_USE_GPU and EASYOCR_GPU_AVAILABLE)

    try:
        _easyocr_reader = easyocr.Reader(
            EASYOCR_LANG_LIST,
            gpu=requested_gpu,
            verbose=bool(EASYOCR_VERBOSE),
        )
        OCR_BACKEND_IN_USE = "easyocr_gpu" if requested_gpu else "easyocr_cpu"
        print(f"[INFO] EasyOCR initialized with backend={OCR_BACKEND_IN_USE}")
        return _easyocr_reader
    except Exception as exc:
        if requested_gpu:
            warnings.warn(
                f"EasyOCR GPU init failed ({exc}). Retrying on CPU.",
                RuntimeWarning,
            )
            try:
                _easyocr_reader = easyocr.Reader(
                    EASYOCR_LANG_LIST,
                    gpu=False,
                    verbose=bool(EASYOCR_VERBOSE),
                )
                OCR_BACKEND_IN_USE = "easyocr_cpu"
                print("[INFO] EasyOCR initialized on CPU fallback")
                return _easyocr_reader
            except Exception as exc2:
                warnings.warn(
                    f"EasyOCR CPU fallback failed ({exc2}).", RuntimeWarning
                )
                return None

        warnings.warn(f"EasyOCR init failed ({exc}).", RuntimeWarning)
        return None


def _extract_bbox_from_easyocr_quad(quad: Any) -> tuple[int, int, int, int] | None:
    try:
        arr = np.asarray(quad, dtype=float)
        if arr.ndim != 2 or arr.shape[1] != 2:
            return None
        x_min = int(np.floor(np.min(arr[:, 0])))
        y_min = int(np.floor(np.min(arr[:, 1])))
        x_max = int(np.ceil(np.max(arr[:, 0])))
        y_max = int(np.ceil(np.max(arr[:, 1])))
        if x_max <= x_min or y_max <= y_min:
            return None
        return (x_min, y_min, x_max, y_max)
    except Exception:
        return None


def _run_easyocr(image: np.ndarray) -> list[OCRBox]:
    reader = _init_easyocr_reader()
    if reader is None:
        return []

    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    try:
        detections = reader.readtext(rgb, detail=1, paragraph=False)
    except Exception as exc:
        warnings.warn(f"EasyOCR inference failed: {exc}", RuntimeWarning)
        return []

    boxes: list[OCRBox] = []
    for item in detections:
        if not isinstance(item, (list, tuple)) or len(item) < 3:
            continue

        quad, text, conf = item[0], str(item[1]).strip(), _safe_float(item[2], default=0.0)
        if not text:
            continue

        bbox = _extract_bbox_from_easyocr_quad(quad)
        if bbox is None:
            continue

        conf_100 = conf * 100.0 if conf <= 1.0 else conf
        conf_100 = float(np.clip(conf_100, 0.0, 100.0))

        boxes.append(
            OCRBox(
                text=text,
                confidence=conf_100,
                bbox=bbox,
            )
        )

    return boxes


def run_tesseract_ocr(image: np.ndarray) -> list[OCRBox]:
    global OCR_BACKEND_IN_USE

    mode = str(OCR_BACKEND).strip().lower()

    if mode == "tesseract_only":
        OCR_BACKEND_IN_USE = "tesseract"
        return _run_tesseract_ocr(image)

    if mode == "easyocr_only":
        boxes = _run_easyocr(image)
        if boxes:
            return boxes
        return []

    # default: easyocr first, then tesseract fallback
    boxes = _run_easyocr(image)
    if boxes:
        return boxes

    t_boxes = _run_tesseract_ocr(image)
    if t_boxes:
        OCR_BACKEND_IN_USE = "tesseract"
    return t_boxes


## 7. Price-tag text heuristics


In [ ]:
PRICE_PATTERN = re.compile(
    r"(?<!\d)(?:\d{1,3}(?:[ \u00A0]\d{3})+|\d{1,5})(?:[., ]\d{2})?(?!\d)"
)
BARCODE_PATTERN = re.compile(r"(?<!\d)\d{8,14}(?!\d)")

KEYWORDS = [
    "карта",
    "цена",
    "скидка",
    "акция",
    "руб",
    "рублей",
    "р",
    "₽",
    "шт",
    "кг",
]


def normalize_text(text: str) -> str:
    t = (text or "").strip().lower().replace("ё", "е")
    t = t.replace(" ", " ")
    t = re.sub(r"\s+", " ", t)
    return t


def is_price_text(text: str) -> bool:
    norm = normalize_text(text)
    if not norm:
        return False

    currency_hint = bool(re.search(r"(₽|руб|\bр\b)", norm))

    for token in PRICE_PATTERN.findall(norm):
        token_clean = token.replace(" ", " ").strip()
        numeric = token_clean.replace(" ", "").replace(",", ".")
        try:
            value = float(numeric)
        except Exception:
            continue

        has_fraction = bool(re.search(r"[., ]\d{2}$", token_clean))

        if has_fraction:
            return True
        if value >= 100:
            return True
        if currency_hint and value >= 10:
            return True

    return False


def is_keyword_text(text: str) -> bool:
    norm = normalize_text(text)
    if not norm:
        return False

    if "₽" in norm:
        return True

    for kw in KEYWORDS:
        if kw == "р":
            if re.search(r"\bр\b", norm):
                return True
            continue
        if kw in norm:
            return True
    return False


def score_ocr_box(box: OCRBox) -> float:
    score = 0.0
    if is_price_text(box.text):
        score += 1.0
    if is_keyword_text(box.text):
        score += 0.5
    score += (max(0.0, min(100.0, box.confidence)) / 100.0) * 0.3
    return float(score)


## 8. Candidate building from OCR boxes


In [ ]:
def box_iou(box_a: tuple[int, int, int, int], box_b: tuple[int, int, int, int]) -> float:
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter_area

    if union <= 0:
        return 0.0
    return float(inter_area / union)


def nms(candidates: list[Candidate], iou_threshold: float = 0.5) -> list[Candidate]:
    if not candidates:
        return []

    order = np.argsort([c.score for c in candidates])[::-1]
    keep: list[Candidate] = []

    while len(order) > 0:
        idx = int(order[0])
        current = candidates[idx]
        keep.append(current)

        remaining = []
        for j in order[1:]:
            j = int(j)
            if box_iou(current.bbox, candidates[j].bbox) < iou_threshold:
                remaining.append(j)

        order = np.array(remaining, dtype=int)

    return keep


def _bbox_center(box: tuple[int, int, int, int]) -> tuple[float, float]:
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)


def _expand_and_clip_bbox(
    box: tuple[int, int, int, int],
    image_shape: tuple[int, ...],
    padding: float,
) -> tuple[int, int, int, int]:
    h, w = image_shape[:2]
    x1, y1, x2, y2 = box
    bw = max(1, x2 - x1)
    bh = max(1, y2 - y1)

    pad_x = int(round(bw * padding))
    pad_y = int(round(bh * padding))

    nx1 = max(0, x1 - pad_x)
    ny1 = max(0, y1 - pad_y)
    nx2 = min(w - 1, x2 + pad_x)
    ny2 = min(h - 1, y2 + pad_y)

    if nx2 <= nx1:
        nx2 = min(w - 1, nx1 + 1)
    if ny2 <= ny1:
        ny2 = min(h - 1, ny1 + 1)

    return (nx1, ny1, nx2, ny2)


def build_price_tag_candidates(
    ocr_boxes: list[OCRBox],
    image_shape: tuple[int, ...],
    padding: float = 0.15,
    seed_score_thr: float = 0.7,
    neighbor_radius_scale: float = 2.0,
    min_area_ratio: float = 0.0005,
    max_area_ratio: float = 0.25,
    aspect_ratio_range: tuple[float, float] = (0.2, 8.0),
    nms_iou: float = 0.5,
) -> list[Candidate]:
    if not ocr_boxes:
        return []

    h, w = image_shape[:2]
    img_area = float(h * w)

    seeds = [
        box
        for box in ocr_boxes
        if (is_price_text(box.text) or is_keyword_text(box.text)) and score_ocr_box(box) >= seed_score_thr
    ]
    if not seeds:
        return []

    candidates: list[Candidate] = []

    for seed in seeds:
        sx1, sy1, sx2, sy2 = seed.bbox
        sw = max(1, sx2 - sx1)
        sh = max(1, sy2 - sy1)
        scx, scy = _bbox_center(seed.bbox)
        radius = max(150.0, neighbor_radius_scale * sw, neighbor_radius_scale * sh)

        group: list[OCRBox] = []
        for box in ocr_boxes:
            cx, cy = _bbox_center(box.bbox)
            dist = math.hypot(cx - scx, cy - scy)
            if dist <= radius and (score_ocr_box(box) > 0.0 or box is seed):
                group.append(box)

        if not group:
            continue

        gx1 = min(b.bbox[0] for b in group)
        gy1 = min(b.bbox[1] for b in group)
        gx2 = max(b.bbox[2] for b in group)
        gy2 = max(b.bbox[3] for b in group)

        bx = _expand_and_clip_bbox((gx1, gy1, gx2, gy2), image_shape, padding)
        x1, y1, x2, y2 = bx

        bw = max(1, x2 - x1)
        bh = max(1, y2 - y1)
        area_ratio = (bw * bh) / img_area
        aspect_ratio = bw / float(max(1, bh))

        if not (min_area_ratio <= area_ratio <= max_area_ratio):
            continue
        if not (aspect_ratio_range[0] <= aspect_ratio <= aspect_ratio_range[1]):
            continue

        group_scores = [score_ocr_box(b) for b in group]
        mean_group_score = float(np.mean(group_scores)) if group_scores else 0.0
        boost = 0.05 * min(len(group), 6)
        cand_score = mean_group_score + boost

        matched_texts = [b.text for b in group if b.text.strip()]
        candidates.append(
            Candidate(
                bbox=bx,
                score=float(cand_score),
                source="ocr_text_regions",
                matched_texts=matched_texts,
            )
        )

    return nms(candidates, iou_threshold=nms_iou)


## 9. Optional fallback without OCR


In [ ]:
def find_rectangular_candidates(
    image: np.ndarray,
    min_area_ratio: float = 0.0005,
    max_area_ratio: float = 0.25,
    aspect_ratio_range: tuple[float, float] = (0.2, 8.0),
    nms_iou: float = 0.4,
) -> list[Candidate]:
    h, w = image.shape[:2]
    img_area = float(h * w)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(blur, 50, 150)
    binary = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        31,
        7,
    )
    mask = cv2.bitwise_or(edges, binary)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    candidates: list[Candidate] = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area <= 0:
            continue

        area_ratio = area / img_area
        if area_ratio < min_area_ratio or area_ratio > max_area_ratio:
            continue

        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.03 * peri, True)

        x, y, bw, bh = cv2.boundingRect(approx if len(approx) >= 4 else cnt)
        if bw <= 2 or bh <= 2:
            continue

        aspect_ratio = bw / float(max(1, bh))
        if aspect_ratio < aspect_ratio_range[0] or aspect_ratio > aspect_ratio_range[1]:
            continue

        rect_fill = area / float(max(1, bw * bh))
        if rect_fill < 0.35:
            continue

        bbox = _expand_and_clip_bbox((x, y, x + bw, y + bh), image.shape, padding=0.05)
        score = 0.25 + 0.5 * min(1.0, rect_fill) + 0.25 * min(1.0, area_ratio / max_area_ratio)

        candidates.append(
            Candidate(
                bbox=bbox,
                score=float(score),
                source="contours_fallback",
                matched_texts=[],
            )
        )

    candidates = sorted(candidates, key=lambda c: c.score, reverse=True)[:60]
    return nms(candidates, iou_threshold=nms_iou)


## 10. Optional Threshold Calibration + evaluation utilities


In [ ]:
def _to_float_series(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
        .str.replace(" ", " ", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def load_gt_annotations(gt_path: str | Path) -> pd.DataFrame:
    gt_path = Path(gt_path)
    if not gt_path.exists():
        raise FileNotFoundError(f"GT file does not exist: {gt_path}")

    gt = pd.read_csv(gt_path, sep=None, engine="python")
    required = ["frame_timestamp", "x_min", "y_min", "x_max", "y_max"]
    missing = [c for c in required if c not in gt.columns]
    if missing:
        raise ValueError(f"GT CSV missing required columns: {missing}")

    gt = gt.copy()
    gt["frame_timestamp"] = _to_float_series(gt["frame_timestamp"])
    gt["x_min"] = _to_float_series(gt["x_min"])
    gt["y_min"] = _to_float_series(gt["y_min"])
    gt["x_max"] = _to_float_series(gt["x_max"])
    gt["y_max"] = _to_float_series(gt["y_max"])

    if "filename" in gt.columns:
        gt["filename"] = gt["filename"].astype(str).str.strip()

    gt = gt.dropna(subset=["frame_timestamp", "x_min", "y_min", "x_max", "y_max"]).reset_index(drop=True)
    return gt


def _map_gt_to_extracted_frames(
    gt_df: pd.DataFrame,
    frame_records: list[dict[str, Any]],
    sample_fps: float,
) -> tuple[pd.DataFrame, int]:
    if gt_df.empty or not frame_records:
        return gt_df.copy(), 0

    frame_ts = np.array([int(r["timestamp_ms"]) for r in frame_records], dtype=np.int64)
    frame_idx = np.array([int(r["frame_index"]) for r in frame_records], dtype=np.int64)

    sample_interval_ms = int(round(1000.0 / max(sample_fps, 1e-6)))
    tolerance_ms = max(1, int(round(0.5 * sample_interval_ms)))

    mapped_rows = []
    for row in gt_df.itertuples(index=False):
        target = int(round(float(row.frame_timestamp)))
        nearest_idx = int(np.argmin(np.abs(frame_ts - target)))
        nearest_ts = int(frame_ts[nearest_idx])
        delta = abs(nearest_ts - target)
        if delta <= tolerance_ms:
            mapped = dict(row._asdict())
            mapped["mapped_frame_index"] = int(frame_idx[nearest_idx])
            mapped["mapped_timestamp_ms"] = nearest_ts
            mapped["timestamp_delta_ms"] = delta
            mapped_rows.append(mapped)

    mapped_df = pd.DataFrame(mapped_rows)
    return mapped_df, tolerance_ms


def _compute_ap_from_flags(tp_flags: list[int], fp_flags: list[int], total_gt: int) -> float:
    if total_gt <= 0:
        return 0.0
    if not tp_flags:
        return 0.0

    tp = np.cumsum(np.array(tp_flags, dtype=float))
    fp = np.cumsum(np.array(fp_flags, dtype=float))

    precision = tp / np.maximum(tp + fp, 1e-12)
    recall = tp / max(float(total_gt), 1e-12)

    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])
    return float(ap)


def evaluate_detections(
    pred_df: pd.DataFrame,
    gt_mapped_df: pd.DataFrame,
    iou_threshold: float = 0.5,
) -> dict[str, float]:
    if gt_mapped_df is None or gt_mapped_df.empty:
        return {
            "precision@0.5": float("nan"),
            "recall@0.5": float("nan"),
            "f1@0.5": float("nan"),
            "mean_iou": float("nan"),
            "ap@0.5": float("nan"),
            "duplicate_rate_eval": float("nan"),
            "tp": 0.0,
            "fp": 0.0,
            "fn": 0.0,
            "gt_count": 0.0,
            "pred_count_eval": float(len(pred_df)),
        }

    gt_by_frame: dict[int, list[tuple[int, int, int, int]]] = {}
    for row in gt_mapped_df.itertuples(index=False):
        fidx = int(row.mapped_frame_index)
        box = (
            int(round(float(row.x_min))),
            int(round(float(row.y_min))),
            int(round(float(row.x_max))),
            int(round(float(row.y_max))),
        )
        gt_by_frame.setdefault(fidx, []).append(box)

    pred_items = []
    if pred_df is not None and not pred_df.empty:
        for row in pred_df.itertuples(index=False):
            pred_items.append(
                {
                    "frame_index": int(row.frame_index),
                    "bbox": (
                        int(row.x_min),
                        int(row.y_min),
                        int(row.x_max),
                        int(row.y_max),
                    ),
                    "score": float(row.candidate_score),
                }
            )

    pred_items = sorted(pred_items, key=lambda x: x["score"], reverse=True)

    matched_flags = {
        fidx: [False] * len(boxes)
        for fidx, boxes in gt_by_frame.items()
    }

    tp_flags: list[int] = []
    fp_flags: list[int] = []
    matched_ious: list[float] = []

    for pred in pred_items:
        fidx = pred["frame_index"]
        gt_boxes = gt_by_frame.get(fidx, [])
        used = matched_flags.get(fidx, [])

        best_iou = 0.0
        best_gt_idx = -1

        for i, gt_box in enumerate(gt_boxes):
            if used[i]:
                continue
            iou = box_iou(pred["bbox"], gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = i

        if best_gt_idx >= 0 and best_iou >= iou_threshold:
            used[best_gt_idx] = True
            tp_flags.append(1)
            fp_flags.append(0)
            matched_ious.append(best_iou)
        else:
            tp_flags.append(0)
            fp_flags.append(1)

    tp = int(sum(tp_flags))
    fp = int(sum(fp_flags))
    gt_count = int(len(gt_mapped_df))
    fn = int(max(0, gt_count - tp))

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, gt_count)
    f1 = 0.0 if (precision + recall) == 0 else (2 * precision * recall) / (precision + recall)
    mean_iou = float(np.mean(matched_ious)) if matched_ious else 0.0
    ap_05 = _compute_ap_from_flags(tp_flags, fp_flags, gt_count)

    dup_sum = 0
    for fidx, gt_boxes in gt_by_frame.items():
        gt_n = len(gt_boxes)
        pred_n = int((pred_df["frame_index"] == fidx).sum()) if pred_df is not None and not pred_df.empty else 0
        dup_sum += max(0, pred_n - gt_n)
    duplicate_rate_eval = dup_sum / max(1, gt_count)

    return {
        "precision@0.5": float(precision),
        "recall@0.5": float(recall),
        "f1@0.5": float(f1),
        "mean_iou": float(mean_iou),
        "ap@0.5": float(ap_05),
        "duplicate_rate_eval": float(duplicate_rate_eval),
        "tp": float(tp),
        "fp": float(fp),
        "fn": float(fn),
        "gt_count": float(gt_count),
        "pred_count_eval": float(len(pred_items)),
    }


def run_optional_threshold_calibration(
    frame_records: list[dict[str, Any]],
    gt_df: pd.DataFrame,
    sample_fps: float,
    base_params: dict[str, float],
) -> tuple[dict[str, float], dict[str, Any], pd.DataFrame | None]:
    if gt_df is None or gt_df.empty:
        return base_params.copy(), {
            "calibration_used": False,
            "reason": "GT dataframe is empty or missing",
        }, None

    mapped_gt, tolerance_ms = _map_gt_to_extracted_frames(gt_df, frame_records, sample_fps)
    if mapped_gt.empty:
        return base_params.copy(), {
            "calibration_used": False,
            "reason": "No GT rows matched extracted timestamps",
            "timestamp_tolerance_ms": tolerance_ms,
        }, mapped_gt

    frame_by_index = {int(r["frame_index"]): r for r in frame_records}
    eval_frame_indices = sorted(mapped_gt["mapped_frame_index"].unique().tolist())

    frame_cache: dict[int, dict[str, Any]] = {}
    for frame_idx in eval_frame_indices:
        rec = frame_by_index.get(int(frame_idx))
        if rec is None:
            continue
        image = cv2.imread(rec["frame_path"])
        if image is None:
            continue
        ocr_boxes = run_tesseract_ocr(image)
        frame_cache[int(frame_idx)] = {
            "image": image,
            "ocr_boxes": ocr_boxes,
        }

    if not frame_cache:
        return base_params.copy(), {
            "calibration_used": False,
            "reason": "Could not load mapped calibration frames",
            "timestamp_tolerance_ms": tolerance_ms,
        }, mapped_gt

    grid = {
        "padding": [0.10, 0.15, 0.20],
        "seed_score_thr": [0.5, 0.7, 0.9],
        "neighbor_radius_scale": [1.5, 2.0, 2.5],
        "nms_iou": [0.4, 0.5, 0.6],
        "min_area_ratio": [0.0003, 0.0005],
        "max_area_ratio": [0.20, 0.25, 0.30],
    }

    keys = list(grid.keys())
    combos = list(itertools.product(*[grid[k] for k in keys]))

    best_params = base_params.copy()
    best_objective = -1e9
    best_metrics: dict[str, float] = {}

    for values in tqdm(combos, desc="Calibrating thresholds"):
        params = base_params.copy()
        for k, v in zip(keys, values):
            params[k] = float(v)

        pred_records = []
        for frame_idx, cached in frame_cache.items():
            image = cached["image"]
            ocr_boxes = cached["ocr_boxes"]

            candidates = build_price_tag_candidates(
                ocr_boxes=ocr_boxes,
                image_shape=image.shape,
                padding=float(params["padding"]),
                seed_score_thr=float(params["seed_score_thr"]),
                neighbor_radius_scale=float(params["neighbor_radius_scale"]),
                min_area_ratio=float(params["min_area_ratio"]),
                max_area_ratio=float(params["max_area_ratio"]),
                aspect_ratio_range=(float(params["aspect_ratio_min"]), float(params["aspect_ratio_max"])),
                nms_iou=float(params["nms_iou"]),
            )

            if not candidates:
                candidates = find_rectangular_candidates(
                    image,
                    min_area_ratio=float(params["min_area_ratio"]),
                    max_area_ratio=float(params["max_area_ratio"]),
                    aspect_ratio_range=(float(params["aspect_ratio_min"]), float(params["aspect_ratio_max"])),
                    nms_iou=float(params["nms_iou"]),
                )

            for cand in candidates:
                x1, y1, x2, y2 = cand.bbox
                pred_records.append(
                    {
                        "frame_index": int(frame_idx),
                        "candidate_score": float(cand.score),
                        "x_min": int(x1),
                        "y_min": int(y1),
                        "x_max": int(x2),
                        "y_max": int(y2),
                    }
                )

        pred_df = pd.DataFrame(pred_records)
        eval_metrics = evaluate_detections(pred_df, mapped_gt, iou_threshold=0.5)

        f1 = float(eval_metrics.get("f1@0.5", 0.0))
        dup = float(eval_metrics.get("duplicate_rate_eval", 0.0))
        objective = f1 - 0.05 * dup

        if objective > best_objective:
            best_objective = objective
            best_params = params.copy()
            best_metrics = eval_metrics.copy()

    summary = {
        "calibration_used": True,
        "timestamp_tolerance_ms": tolerance_ms,
        "eval_frames": len(frame_cache),
        "eval_gt_boxes": int(len(mapped_gt)),
        "objective": float(best_objective),
        "best_metrics": best_metrics,
        "best_params": best_params,
    }

    return best_params, summary, mapped_gt


## 11. Inference pipeline and metrics summary


In [ ]:
def _is_border_cut(
    bbox: tuple[int, int, int, int],
    image_shape: tuple[int, ...],
    margin: int = 1,
) -> bool:
    x1, y1, x2, y2 = bbox
    h, w = image_shape[:2]
    return x1 <= margin or y1 <= margin or x2 >= (w - 1 - margin) or y2 >= (h - 1 - margin)


def _draw_candidates(image: np.ndarray, candidates: list[Candidate]) -> np.ndarray:
    vis = image.copy()
    for cand in candidates:
        x1, y1, x2, y2 = cand.bbox
        color = (0, 255, 0) if cand.source == "ocr_text_regions" else (0, 165, 255)
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
        label = f"{cand.source} | {cand.score:.2f}"
        cv2.putText(
            vis,
            label,
            (x1, max(0, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            1,
            cv2.LINE_AA,
        )
    return vis


def _apply_rotation(image: np.ndarray, rotation: str) -> np.ndarray:
    rot = (rotation or "none").lower()
    if rot == "cw":
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    if rot == "ccw":
        return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if rot == "180":
        return cv2.rotate(image, cv2.ROTATE_180)
    return image


def _resolve_rotation_mode(
    frame_records: list[dict[str, Any]],
    requested_mode: str,
) -> str:
    mode = (requested_mode or "none").strip().lower()
    if mode in {"none", "cw", "ccw", "180"}:
        return mode
    # For this dataset we default to fixed CCW rotation.
    return "ccw"


def _rotate_point(x: float, y: float, width: float, height: float, rotation: str) -> tuple[float, float]:
    rot = (rotation or "none").lower()
    if rot == "cw":
        return (height - 1.0 - y, x)
    if rot == "ccw":
        return (y, width - 1.0 - x)
    if rot == "180":
        return (width - 1.0 - x, height - 1.0 - y)
    return (x, y)


def _rotate_bbox(
    bbox: tuple[float, float, float, float],
    width: float,
    height: float,
    rotation: str,
) -> tuple[float, float, float, float]:
    x1, y1, x2, y2 = bbox
    corners = [
        (x1, y1),
        (x2, y1),
        (x2, y2),
        (x1, y2),
    ]
    rc = [_rotate_point(x, y, width, height, rotation) for (x, y) in corners]
    xs = [p[0] for p in rc]
    ys = [p[1] for p in rc]
    return (min(xs), min(ys), max(xs), max(ys))


def _transform_gt_for_rotation(
    gt_df: pd.DataFrame,
    rotation: str,
    orig_width: int,
    orig_height: int,
) -> pd.DataFrame:
    rot = (rotation or "none").lower()
    if gt_df is None or gt_df.empty or rot == "none":
        return gt_df

    out = gt_df.copy()
    x1 = _to_float_series(out["x_min"])
    y1 = _to_float_series(out["y_min"])
    x2 = _to_float_series(out["x_max"])
    y2 = _to_float_series(out["y_max"])

    new_boxes = [
        _rotate_bbox((a, b, c, d), float(orig_width), float(orig_height), rot)
        for a, b, c, d in zip(x1, y1, x2, y2)
    ]

    out["x_min"] = [b[0] for b in new_boxes]
    out["y_min"] = [b[1] for b in new_boxes]
    out["x_max"] = [b[2] for b in new_boxes]
    out["y_max"] = [b[3] for b in new_boxes]
    return out


def _estimate_unique_price_tags(
    df: pd.DataFrame,
    iou_threshold: float = 0.5,
    max_gap_ms: int = 2000,
) -> tuple[int, list[dict[str, Any]]]:
    if df is None or df.empty:
        return 0, []

    work = df.sort_values(["timestamp_ms", "candidate_score"], ascending=[True, False]).reset_index(drop=True)
    tracks: list[dict[str, Any]] = []

    for row in work.itertuples(index=False):
        ts = int(row.timestamp_ms)
        bbox = (int(row.x_min), int(row.y_min), int(row.x_max), int(row.y_max))

        assigned = False
        for track in reversed(tracks):
            if ts - track["last_ts"] > max_gap_ms:
                continue
            if box_iou(bbox, track["bbox"]) >= iou_threshold:
                track["bbox"] = bbox
                track["last_ts"] = ts
                track["count"] += 1
                assigned = True
                break

        if not assigned:
            tracks.append({"bbox": bbox, "last_ts": ts, "count": 1})

    return len(tracks), tracks


def _compute_proxy_and_business_metrics(
    result_df: pd.DataFrame,
    frame_stats: list[dict[str, Any]],
    processing_time_sec: float,
) -> dict[str, float]:
    frames_total = len(frame_stats)
    if frames_total == 0:
        return {
            "ocr_valid_rate": 0.0,
            "price_read_rate": 0.0,
            "barcode_read_rate": 0.0,
            "candidates_per_frame_mean": 0.0,
            "mean_quality_score": 0.0,
            "mean_sharpness": 0.0,
            "border_cut_rate": 0.0,
            "fallback_usage_rate": 0.0,
            "no_detection_frame_rate": 1.0,
            "processing_time_sec": float(processing_time_sec),
            "effective_fps": 0.0,
            "unique_price_tags_est": 0.0,
            "duplicate_rate_video": 0.0,
            "actionable_frame_rate": 0.0,
            "frame_sharpness_score_mean": 0.0,
            "frame_custom_score_mean": 0.0,
        }

    total_ocr_boxes = sum(int(s["ocr_box_count"]) for s in frame_stats)
    total_price_boxes = sum(int(s["price_box_count"]) for s in frame_stats)
    total_barcode_boxes = sum(int(s["barcode_box_count"]) for s in frame_stats)
    total_candidates = sum(int(s["detections"]) for s in frame_stats)

    ocr_valid_rate = float(np.mean([1.0 if s["ocr_box_count"] > 0 else 0.0 for s in frame_stats]))
    fallback_usage_rate = float(np.mean([1.0 if s["fallback_used"] else 0.0 for s in frame_stats]))
    no_detection_frame_rate = float(np.mean([1.0 if s["detections"] == 0 else 0.0 for s in frame_stats]))

    unique_est, _ = _estimate_unique_price_tags(result_df, iou_threshold=0.5, max_gap_ms=2000)
    duplicate_rate_video = max(0.0, (float(total_candidates) - float(unique_est)) / max(1.0, float(unique_est)))

    metrics = {
        "ocr_valid_rate": ocr_valid_rate,
        "price_read_rate": float(total_price_boxes / max(1, total_ocr_boxes)),
        "barcode_read_rate": float(total_barcode_boxes / max(1, total_ocr_boxes)),
        "candidates_per_frame_mean": float(total_candidates / max(1, frames_total)),
        "mean_quality_score": float(result_df["quality_score"].mean()) if not result_df.empty else 0.0,
        "mean_sharpness": float(result_df["sharpness"].mean()) if not result_df.empty else 0.0,
        "border_cut_rate": float(result_df["border_cut"].mean()) if (not result_df.empty and "border_cut" in result_df.columns) else 0.0,
        "fallback_usage_rate": fallback_usage_rate,
        "no_detection_frame_rate": no_detection_frame_rate,
        "processing_time_sec": float(processing_time_sec),
        "effective_fps": float(frames_total / max(processing_time_sec, 1e-9)),
        "unique_price_tags_est": float(unique_est),
        "duplicate_rate_video": float(duplicate_rate_video),
        "actionable_frame_rate": float(1.0 - no_detection_frame_rate),
        "frame_sharpness_score_mean": float(np.mean([s["frame_sharpness_score"] for s in frame_stats])) if frame_stats else 0.0,
        "frame_custom_score_mean": float(np.mean([s["frame_custom_score"] for s in frame_stats])) if frame_stats else 0.0,
    }
    return metrics


def process_video_baseline(
    video_path: str,
    output_dir: str,
    sample_fps: float = 2.0,
    max_frames: int | None = None,
    params: dict[str, float] | None = None,
    gt_df: pd.DataFrame | None = None,
    enable_calibration: bool = True,
    frame_rotation: str = "none",
) -> tuple[pd.DataFrame, dict[str, Any]]:
    t0 = time.perf_counter()

    params_final = DEFAULT_HEURISTIC_PARAMS.copy()
    if params:
        params_final.update(params)

    out_dir = Path(output_dir)
    frames_dir = out_dir / "frames"
    crops_dir = out_dir / "crops"
    debug_dir = out_dir / "debug_frames"
    preproc_dir = out_dir / "preprocessed_frames"

    out_dir.mkdir(parents=True, exist_ok=True)
    frames_dir.mkdir(parents=True, exist_ok=True)
    crops_dir.mkdir(parents=True, exist_ok=True)
    debug_dir.mkdir(parents=True, exist_ok=True)
    preproc_dir.mkdir(parents=True, exist_ok=True)

    video_info = inspect_video(video_path)
    print("Video info:")
    for k, v in video_info.items():
        print(f"  {k}: {v}")

    frame_records = extract_frames(
        video_path=video_path,
        frames_dir=frames_dir,
        sample_fps=sample_fps,
        max_frames=max_frames,
    )
    print(f"Extracted frames: {len(frame_records)}")

    resolved_rotation = _resolve_rotation_mode(frame_records, frame_rotation)
    print(f"Frame rotation: requested={frame_rotation}, resolved={resolved_rotation}")

    gt_for_eval = gt_df
    if gt_df is not None and not gt_df.empty and resolved_rotation != "none":
        gt_for_eval = _transform_gt_for_rotation(
            gt_df=gt_df,
            rotation=resolved_rotation,
            orig_width=int(video_info.get("width", 0) or 0),
            orig_height=int(video_info.get("height", 0) or 0),
        )

    calibration_summary: dict[str, Any] = {
        "calibration_used": False,
        "reason": "GT not provided",
    }
    gt_mapped_df: pd.DataFrame | None = None

    if enable_calibration and gt_for_eval is not None and not gt_for_eval.empty:
        print("Running optional threshold calibration...")
        best_params, calibration_summary, gt_mapped_df = run_optional_threshold_calibration(
            frame_records=frame_records,
            gt_df=gt_for_eval,
            sample_fps=sample_fps,
            base_params=params_final,
        )
        params_final.update(best_params)
        print("Calibration summary:")
        print(json.dumps(calibration_summary, ensure_ascii=False, indent=2, default=str))
    else:
        if not enable_calibration:
            print("Calibration disabled by config. Using default heuristic parameters.")
        else:
            print("GT annotations not provided. Using default heuristic parameters.")

    if gt_mapped_df is None and gt_for_eval is not None and not gt_for_eval.empty:
        gt_mapped_df, _ = _map_gt_to_extracted_frames(gt_for_eval, frame_records, sample_fps)

    debug_records: list[dict[str, Any]] = []
    frame_stats: list[dict[str, Any]] = []
    frame_quality_records: list[dict[str, Any]] = []

    video_filename = Path(video_path).name

    for rec in tqdm(frame_records, desc="Inference"):
        frame_idx = int(rec["frame_index"])
        timestamp_ms = int(rec["timestamp_ms"])
        frame_path = rec["frame_path"]

        image = cv2.imread(frame_path)
        if image is None:
            continue

        image_rot = _apply_rotation(image, resolved_rotation)
        prep = preprocess_frame_bundle(image_rot)
        ocr_image = prep["ocr_ready_bgr"]

        raw_scores = compute_frame_scores(image_rot)
        prep_scores = compute_frame_scores(ocr_image)

        rotated_path = preproc_dir / f"rotated_{frame_idx:08d}_{timestamp_ms:010d}ms.jpg"
        preproc_path = preproc_dir / f"preproc_{frame_idx:08d}_{timestamp_ms:010d}ms.jpg"
        if SAVE_PREPROCESSED_FRAMES:
            cv2.imwrite(str(rotated_path), image_rot)
            cv2.imwrite(str(preproc_path), ocr_image)

        frame_quality_records.append(
            {
                "video_filename": video_filename,
                "frame_index": frame_idx,
                "timestamp_ms": timestamp_ms,
                "frame_path": frame_path,
                "rotated_frame_path": str(rotated_path),
                "preprocessed_frame_path": str(preproc_path),
                "raw_sharpness": raw_scores["sharpness"],
                "raw_brightness": raw_scores["brightness"],
                "raw_contrast": raw_scores["contrast"],
                "prep_sharpness": prep_scores["sharpness"],
                "prep_brightness": prep_scores["brightness"],
                "prep_contrast": prep_scores["contrast"],
                "frame_sharpness_score": prep_scores["sharpness_score"],
                "frame_custom_score": prep_scores["custom_frame_score"],
            }
        )

        ocr_boxes = run_tesseract_ocr(ocr_image)
        price_box_count = sum(1 for b in ocr_boxes if is_price_text(b.text))
        barcode_box_count = sum(1 for b in ocr_boxes if BARCODE_PATTERN.search(normalize_text(b.text)) is not None)

        candidates = build_price_tag_candidates(
            ocr_boxes=ocr_boxes,
            image_shape=image_rot.shape,
            padding=float(params_final["padding"]),
            seed_score_thr=float(params_final["seed_score_thr"]),
            neighbor_radius_scale=float(params_final["neighbor_radius_scale"]),
            min_area_ratio=float(params_final["min_area_ratio"]),
            max_area_ratio=float(params_final["max_area_ratio"]),
            aspect_ratio_range=(float(params_final["aspect_ratio_min"]), float(params_final["aspect_ratio_max"])),
            nms_iou=float(params_final["nms_iou"]),
        )

        fallback_used = False
        if not candidates:
            candidates = find_rectangular_candidates(
                image_rot,
                min_area_ratio=float(params_final["min_area_ratio"]),
                max_area_ratio=float(params_final["max_area_ratio"]),
                aspect_ratio_range=(float(params_final["aspect_ratio_min"]), float(params_final["aspect_ratio_max"])),
                nms_iou=float(params_final["nms_iou"]),
            )
            fallback_used = len(candidates) > 0

        debug_img = _draw_candidates(image_rot, candidates)
        debug_frame_path = debug_dir / f"debug_{frame_idx:08d}_{timestamp_ms:010d}ms.jpg"
        cv2.imwrite(str(debug_frame_path), debug_img)

        for cand_id, cand in enumerate(candidates):
            x1, y1, x2, y2 = cand.bbox
            crop = image_rot[y1:y2, x1:x2]
            if crop is None or crop.size == 0:
                continue

            crop_filename = f"crop_{frame_idx:08d}_{timestamp_ms:010d}ms_{cand_id:03d}.jpg"
            crop_path = crops_dir / crop_filename
            cv2.imwrite(str(crop_path), crop)

            quality = compute_crop_quality(crop, candidate_score=float(cand.score))
            border_cut = _is_border_cut(cand.bbox, image_rot.shape)

            debug_records.append(
                {
                    "video_filename": video_filename,
                    "frame_index": frame_idx,
                    "timestamp_ms": timestamp_ms,
                    "candidate_id": cand_id,
                    "crop_path": str(crop_path),
                    "debug_frame_path": str(debug_frame_path),
                    "rotated_frame_path": str(rotated_path),
                    "preprocessed_frame_path": str(preproc_path),
                    "x_min": int(x1),
                    "y_min": int(y1),
                    "x_max": int(x2),
                    "y_max": int(y2),
                    "candidate_score": float(cand.score),
                    "quality_score": float(quality["quality_score"]),
                    "sharpness": float(quality["sharpness"]),
                    "brightness": float(quality["brightness"]),
                    "contrast": float(quality["contrast"]),
                    "matched_texts": " | ".join(cand.matched_texts) if cand.matched_texts else "",
                    "source": cand.source,
                    "border_cut": int(border_cut),
                    "frame_sharpness_score": prep_scores["sharpness_score"],
                    "frame_custom_score": prep_scores["custom_frame_score"],
                }
            )

        frame_stats.append(
            {
                "frame_index": frame_idx,
                "timestamp_ms": timestamp_ms,
                "ocr_box_count": len(ocr_boxes),
                "price_box_count": int(price_box_count),
                "barcode_box_count": int(barcode_box_count),
                "detections": len(candidates),
                "fallback_used": bool(fallback_used),
                "frame_sharpness_score": prep_scores["sharpness_score"],
                "frame_custom_score": prep_scores["custom_frame_score"],
            }
        )

    result_df = pd.DataFrame(debug_records)

    required_columns = [
        "video_filename",
        "frame_index",
        "timestamp_ms",
        "candidate_id",
        "crop_path",
        "debug_frame_path",
        "x_min",
        "y_min",
        "x_max",
        "y_max",
        "candidate_score",
        "quality_score",
        "sharpness",
        "brightness",
        "contrast",
        "matched_texts",
        "source",
        "frame_sharpness_score",
        "frame_custom_score",
    ]
    for col in required_columns:
        if col not in result_df.columns:
            result_df[col] = []

    if not result_df.empty:
        result_df["rank_by_quality"] = result_df.groupby("frame_index")["quality_score"].rank(ascending=False, method="first")
        result_df["rank_by_sharpness"] = result_df.groupby("frame_index")["sharpness"].rank(ascending=False, method="first")
        result_df["rank_by_candidate"] = result_df.groupby("frame_index")["candidate_score"].rank(ascending=False, method="first")
        result_df["is_best_by_quality"] = (result_df["rank_by_quality"] == 1).astype(int)
        result_df["is_best_by_sharpness"] = (result_df["rank_by_sharpness"] == 1).astype(int)
        result_df["is_best_by_candidate"] = (result_df["rank_by_candidate"] == 1).astype(int)

    debug_csv_path = out_dir / "debug_candidates.csv"
    result_df.to_csv(debug_csv_path, index=False)

    frame_quality_df = pd.DataFrame(frame_quality_records)
    frame_quality_csv_path = out_dir / "frame_quality_debug.csv"
    frame_quality_df.to_csv(frame_quality_csv_path, index=False)

    elapsed = time.perf_counter() - t0

    proxy_business_metrics = _compute_proxy_and_business_metrics(
        result_df=result_df,
        frame_stats=frame_stats,
        processing_time_sec=elapsed,
    )

    if gt_mapped_df is not None and not gt_mapped_df.empty:
        eval_pred_df = result_df[
            ["frame_index", "candidate_score", "x_min", "y_min", "x_max", "y_max"]
        ].copy() if not result_df.empty else pd.DataFrame(columns=["frame_index", "candidate_score", "x_min", "y_min", "x_max", "y_max"])
        full_metrics = evaluate_detections(eval_pred_df, gt_mapped_df, iou_threshold=0.5)
    else:
        full_metrics = {
            "precision@0.5": float("nan"),
            "recall@0.5": float("nan"),
            "f1@0.5": float("nan"),
            "mean_iou": float("nan"),
            "ap@0.5": float("nan"),
            "duplicate_rate_eval": float("nan"),
            "tp": float("nan"),
            "fp": float("nan"),
            "fn": float("nan"),
            "gt_count": float("nan"),
            "pred_count_eval": float("nan"),
        }

    metrics_summary: dict[str, Any] = {
        "video_path": video_path,
        "output_dir": str(out_dir),
        "frame_rotation_requested": frame_rotation,
        "frame_rotation_resolved": resolved_rotation,
        "ocr_backend": OCR_BACKEND_IN_USE,
        "frames_processed": len(frame_stats),
        "detections_total": int(len(result_df)),
        "heuristic_params": params_final,
        "calibration": calibration_summary,
        "full_detection_metrics": full_metrics,
        "proxy_business_metrics": proxy_business_metrics,
    }

    flat_metrics = {
        "frames_processed": float(metrics_summary["frames_processed"]),
        "detections_total": float(metrics_summary["detections_total"]),
    }
    for k, v in full_metrics.items():
        flat_metrics[k] = float(v) if pd.notna(v) else np.nan
    for k, v in proxy_business_metrics.items():
        flat_metrics[k] = float(v)

    metrics_df = pd.DataFrame(
        [{"metric": k, "value": v} for k, v in flat_metrics.items()]
    )

    metrics_csv_path = out_dir / "metrics_summary.csv"
    metrics_json_path = out_dir / "metrics_summary.json"
    metrics_df.to_csv(metrics_csv_path, index=False)

    with metrics_json_path.open("w", encoding="utf-8") as f:
        json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

    return result_df, {
        "metrics_summary": metrics_summary,
        "metrics_df": metrics_df,
        "debug_csv_path": str(debug_csv_path),
        "metrics_csv_path": str(metrics_csv_path),
        "metrics_json_path": str(metrics_json_path),
        "frame_quality_csv_path": str(frame_quality_csv_path),
        "output_dir": str(out_dir),
    }


## 12. Run baseline and Metrics Summary


In [ ]:
def discover_video_jobs(
    data_root: str,
    case_filter: str | None = None,
    include_unlabeled: bool = True,
) -> list[dict[str, str | None]]:
    root = Path(data_root)
    if not root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {data_root}")

    jobs: list[dict[str, str | None]] = []
    cf = case_filter.lower().strip() if isinstance(case_filter, str) and case_filter.strip() else None

    for mp4_path in sorted(root.rglob("*.mp4")):
        if any(part.startswith("baseline_ocr_candidates") for part in mp4_path.parts):
            continue

        if not include_unlabeled and any(part.lower() == "unlabeled" for part in mp4_path.parts):
            continue

        path_str = str(mp4_path).lower()
        if cf and cf not in path_str:
            continue

        gt_candidate = mp4_path.with_suffix(".csv")
        gt_path = str(gt_candidate) if gt_candidate.exists() else None

        case_id = f"{mp4_path.parent.name}__{mp4_path.stem}" if mp4_path.parent.name else mp4_path.stem
        jobs.append(
            {
                "case_id": case_id,
                "video_path": str(mp4_path),
                "gt_path": gt_path,
            }
        )

    return jobs


def _safe_case_name(text: str) -> str:
    name = re.sub(r"[^a-zA-Z0-9_.-]+", "_", text)
    return name.strip("_") or "case"


run_mode = str(RUN_MODE).strip().lower()
if run_mode not in {"single", "batch"}:
    raise ValueError(f"RUN_MODE must be 'single' or 'batch', got: {RUN_MODE}")

jobs: list[dict[str, str | None]]
if run_mode == "single":
    if not INPUT_VIDEO_PATH:
        raise ValueError("INPUT_VIDEO_PATH is empty in single mode")
    jobs = [
        {
            "case_id": _safe_case_name(Path(INPUT_VIDEO_PATH).stem),
            "video_path": INPUT_VIDEO_PATH,
            "gt_path": GT_ANNOTATIONS_PATH,
        }
    ]
else:
    jobs = discover_video_jobs(
        data_root=DATA_ROOT,
        case_filter=CASE_FILTER,
        include_unlabeled=INCLUDE_UNLABELED,
    )

if not jobs:
    raise RuntimeError("No video jobs found. Check DATA_ROOT / CASE_FILTER / RUN_MODE.")

print(f"RUN_MODE={run_mode} | jobs found: {len(jobs)}")
for j in jobs:
    print(f"  - {j['case_id']}: {j['video_path']} | GT={j['gt_path']}")

all_run_infos: list[dict[str, Any]] = []
all_result_parts: list[pd.DataFrame] = []
all_frame_quality_parts: list[pd.DataFrame] = []
summary_rows: list[dict[str, Any]] = []

for idx, job in enumerate(jobs, start=1):
    case_id = str(job["case_id"])
    video_path = str(job["video_path"])
    gt_path = job.get("gt_path")

    print("\n" + "=" * 80)
    print(f"[{idx}/{len(jobs)}] Processing case: {case_id}")

    gt_annotations_df = None
    if gt_path:
        try:
            gt_annotations_df = load_gt_annotations(gt_path)
            print(f"GT rows loaded: {len(gt_annotations_df)}")
        except Exception as exc:
            print(f"[WARN] Failed to load GT annotations ({gt_path}): {exc}")
            gt_annotations_df = None

    if run_mode == "single":
        case_output_dir = OUTPUT_DIR
    else:
        case_output_dir = str(Path(BATCH_OUTPUT_ROOT) / f"baseline_ocr_candidates_{_safe_case_name(case_id)}")

    case_result_df, case_run_info = process_video_baseline(
        video_path=video_path,
        output_dir=case_output_dir,
        sample_fps=SAMPLE_FPS,
        max_frames=MAX_FRAMES,
        params=DEFAULT_HEURISTIC_PARAMS,
        gt_df=gt_annotations_df,
        enable_calibration=ENABLE_CALIBRATION,
        frame_rotation=FRAME_ROTATION_MODE,
    )

    case_result_df = case_result_df.copy()
    if not case_result_df.empty:
        case_result_df["case_id"] = case_id
        case_result_df["video_path"] = video_path
        all_result_parts.append(case_result_df)

    fq_path = case_run_info.get("frame_quality_csv_path")
    if fq_path and Path(fq_path).exists():
        fq_df = pd.read_csv(fq_path)
        fq_df["case_id"] = case_id
        fq_df["video_path"] = video_path
        all_frame_quality_parts.append(fq_df)

    metrics_summary = case_run_info.get("metrics_summary", {})
    full = metrics_summary.get("full_detection_metrics", {})
    proxy = metrics_summary.get("proxy_business_metrics", {})
    summary_rows.append(
        {
            "case_id": case_id,
            "video_path": video_path,
            "gt_used": bool(gt_annotations_df is not None and not gt_annotations_df.empty),
            "output_dir": case_run_info.get("output_dir", case_output_dir),
            "frames_processed": metrics_summary.get("frames_processed", np.nan),
            "detections_total": metrics_summary.get("detections_total", np.nan),
            "precision@0.5": full.get("precision@0.5", np.nan),
            "recall@0.5": full.get("recall@0.5", np.nan),
            "f1@0.5": full.get("f1@0.5", np.nan),
            "ap@0.5": full.get("ap@0.5", np.nan),
            "mean_iou": full.get("mean_iou", np.nan),
            "duplicate_rate_eval": full.get("duplicate_rate_eval", np.nan),
            "ocr_valid_rate": proxy.get("ocr_valid_rate", np.nan),
            "price_read_rate": proxy.get("price_read_rate", np.nan),
            "barcode_read_rate": proxy.get("barcode_read_rate", np.nan),
            "candidates_per_frame_mean": proxy.get("candidates_per_frame_mean", np.nan),
            "mean_quality_score": proxy.get("mean_quality_score", np.nan),
            "mean_sharpness": proxy.get("mean_sharpness", np.nan),
            "frame_sharpness_score_mean": proxy.get("frame_sharpness_score_mean", np.nan),
            "frame_custom_score_mean": proxy.get("frame_custom_score_mean", np.nan),
            "fallback_usage_rate": proxy.get("fallback_usage_rate", np.nan),
            "processing_time_sec": proxy.get("processing_time_sec", np.nan),
            "effective_fps": proxy.get("effective_fps", np.nan),
        }
    )

    all_run_infos.append(case_run_info)

result_df = pd.concat(all_result_parts, ignore_index=True) if all_result_parts else pd.DataFrame()
frame_quality_df_all = pd.concat(all_frame_quality_parts, ignore_index=True) if all_frame_quality_parts else pd.DataFrame()
batch_summary_df = pd.DataFrame(summary_rows)

print("\nBatch summary shape:", batch_summary_df.shape)
display(batch_summary_df)

print("\nCombined result_df shape:", result_df.shape)
display(result_df.head())

print("\nCombined frame_quality_df_all shape:", frame_quality_df_all.shape)
display(frame_quality_df_all.head())

# Keep backward-compatible variables
run_info = all_run_infos[-1] if all_run_infos else {}
metrics_df = batch_summary_df


## 13. Visual inspection


In [ ]:
def _read_rgb(path: str) -> np.ndarray | None:
    img = cv2.imread(path)
    if img is None:
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def show_image_grid_with_titles(
    paths: list[str],
    titles: list[str] | None,
    title: str,
    cols: int = 4,
) -> None:
    if not paths:
        print(f"No images to display for: {title}")
        return

    rows = int(math.ceil(len(paths) / cols))
    plt.figure(figsize=(4.5 * cols, 3.8 * rows))
    for i, p in enumerate(paths, start=1):
        img = _read_rgb(p)
        plt.subplot(rows, cols, i)
        if img is None:
            plt.text(0.5, 0.5, "Image not found", ha="center", va="center")
            plt.axis("off")
            continue
        plt.imshow(img)
        cap = Path(p).name
        if titles and i - 1 < len(titles):
            cap = titles[i - 1]
        plt.title(cap, fontsize=9)
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_preprocess_before_after(frame_paths: list[str], n: int = 6) -> None:
    if not frame_paths:
        print("No frame paths for preprocess preview.")
        return

    sampled = frame_paths[:n]
    for fp in sampled:
        img = cv2.imread(fp)
        if img is None:
            continue
        rot = _apply_rotation(img, FRAME_ROTATION_MODE)
        prep = preprocess_frame_bundle(rot)

        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        axes[0].imshow(cv2.cvtColor(rot, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Rotated input")
        axes[1].imshow(prep["clahe"], cmap="gray")
        axes[1].set_title("CLAHE")
        axes[2].imshow(prep["denoised"], cmap="gray")
        axes[2].set_title("Denoised")
        axes[3].imshow(prep["sharpen"], cmap="gray")
        axes[3].set_title("Sharpened (OCR input)")
        for ax in axes:
            ax.axis("off")
        fig.suptitle(f"Preprocessing chain: {Path(fp).name}")
        plt.tight_layout()
        plt.show()


def top_paths_by_score(df: pd.DataFrame, path_col: str, score_col: str, top_n: int) -> tuple[list[str], list[str]]:
    if df.empty or path_col not in df.columns or score_col not in df.columns:
        return [], []
    work = df.sort_values(score_col, ascending=False).head(top_n)
    paths = work[path_col].astype(str).tolist()
    titles = [f"{score_col}={v:.4f}" for v in work[score_col].astype(float).tolist()]
    return paths, titles


if result_df.empty:
    print("No candidates were produced. Check INPUT_VIDEO_PATH, SAMPLE_FPS, OCR availability, and fallback settings.")
else:
    print("Step A. Show sampled extracted frames (after video slicing)")
    first_case_output_dir = None
    if "batch_summary_df" in globals() and isinstance(batch_summary_df, pd.DataFrame) and not batch_summary_df.empty:
        first_case_output_dir = str(batch_summary_df.iloc[0]["output_dir"])
    elif "run_info" in globals() and run_info:
        first_case_output_dir = run_info.get("output_dir")

    sampled_frame_paths = []
    if first_case_output_dir:
        frame_dir = Path(first_case_output_dir) / "frames"
        sampled_frame_paths = [str(p) for p in sorted(frame_dir.glob("*.jpg"))[:TOP_N_DEBUG_FRAMES]]

    show_image_grid_with_titles(sampled_frame_paths, None, "Extracted frames (raw from video)", cols=3)

    print("Step B. Preprocessing before/after snapshots")
    show_preprocess_before_after(sampled_frame_paths, n=min(PREPROCESS_PREVIEW_N, len(sampled_frame_paths)))

    print("Step C. Frame ranking by different frame-level scores")
    if "frame_quality_df_all" in globals() and isinstance(frame_quality_df_all, pd.DataFrame) and not frame_quality_df_all.empty:
        p1, t1 = top_paths_by_score(frame_quality_df_all, "rotated_frame_path", "frame_sharpness_score", TOP_N_DEBUG_FRAMES)
        show_image_grid_with_titles(p1, t1, f"Top-{TOP_N_DEBUG_FRAMES} frames by frame_sharpness_score", cols=3)

        p2, t2 = top_paths_by_score(frame_quality_df_all, "rotated_frame_path", "frame_custom_score", TOP_N_DEBUG_FRAMES)
        show_image_grid_with_titles(p2, t2, f"Top-{TOP_N_DEBUG_FRAMES} frames by frame_custom_score", cols=3)
    else:
        print("frame_quality_df_all is empty, skip frame score comparison")

    print("Step D. Candidate debug frames with bbox")
    debug_paths = (
        result_df["debug_frame_path"]
        .dropna()
        .drop_duplicates()
        .head(TOP_N_DEBUG_FRAMES)
        .tolist()
    )
    show_image_grid_with_titles(debug_paths, None, "Debug frames with candidate bboxes", cols=3)

    print("Step E. Top-N crops by different crop-level scores")
    pq, tq = top_paths_by_score(result_df, "crop_path", "quality_score", TOP_N_CROPS)
    show_image_grid_with_titles(pq, tq, f"Top-{TOP_N_CROPS} crops by quality_score", cols=4)

    ps, ts = top_paths_by_score(result_df, "crop_path", "sharpness", TOP_N_CROPS)
    show_image_grid_with_titles(ps, ts, f"Top-{TOP_N_CROPS} crops by sharpness", cols=4)

    pc, tc = top_paths_by_score(result_df, "crop_path", "candidate_score", TOP_N_CROPS)
    show_image_grid_with_titles(pc, tc, f"Top-{TOP_N_CROPS} crops by candidate_score", cols=4)

    print("Step F. Random crop audit (manual suspicious check)")
    sample_n = min(TOP_N_CROPS, len(result_df))
    rand_df = result_df.sample(n=sample_n, random_state=RANDOM_SEED) if sample_n > 0 else pd.DataFrame()
    rand_paths = rand_df["crop_path"].astype(str).tolist() if not rand_df.empty else []
    rand_titles = [
        f"q={r.quality_score:.3f} | sh={r.sharpness:.1f} | cs={r.candidate_score:.2f}"
        for r in rand_df.itertuples(index=False)
    ] if not rand_df.empty else []
    show_image_grid_with_titles(rand_paths, rand_titles, f"Random {sample_n} crops for sanity check", cols=4)

    print("Step G. Best crop selection vs rejected candidates (same frame)")
    if all(col in result_df.columns for col in ["rank_by_quality", "rank_by_sharpness", "rank_by_candidate"]):
        multi_df = result_df.groupby("frame_index").filter(lambda g: len(g) >= 3)
        frame_ids = multi_df["frame_index"].drop_duplicates().head(4).tolist()
        compare_paths: list[str] = []
        compare_titles: list[str] = []

        for fid in frame_ids:
            g = multi_df[multi_df["frame_index"] == fid].copy()
            if g.empty:
                continue

            q_best = g.sort_values("quality_score", ascending=False).iloc[0]
            s_best = g.sort_values("sharpness", ascending=False).iloc[0]
            c_best = g.sort_values("candidate_score", ascending=False).iloc[0]
            rejected = g.sort_values("quality_score", ascending=True).iloc[0]

            entries = [
                (q_best, "BEST quality"),
                (s_best, "BEST sharpness"),
                (c_best, "BEST candidate"),
                (rejected, "REJECT (lowest quality)"),
            ]
            for row, tag in entries:
                compare_paths.append(str(row.crop_path))
                compare_titles.append(
                    f"f={fid} {tag}\nq={row.quality_score:.3f} sh={row.sharpness:.1f} cs={row.candidate_score:.2f}"
                )

        show_image_grid_with_titles(compare_paths, compare_titles, "Selection strategy comparison: keep vs reject", cols=4)
    else:
        print("Rank columns not found, skip selection strategy comparison")

    print("Step H. Future improvements and expected effect")
    improvements = pd.DataFrame(
        [
            {"idea": "Temporal tracking across frames", "expected_effect": "-20%..-40% duplicate_rate_video, more stable best-crop choice"},
            {"idea": "Small detector model (YOLO/RT-DETR) before OCR", "expected_effect": "+10%..+25% recall on partially occluded tags"},
            {"idea": "QR decoding branch before OCR text heuristics", "expected_effect": "+barcode_read_rate and better business matching"},
            {"idea": "Perspective rectification for angled tags", "expected_effect": "+5%..+15% OCR quality on skewed labels"},
            {"idea": "Learned crop scoring head", "expected_effect": "better top-N precision than fixed quality formula"},
        ]
    )
    display(improvements)


## 14. Save artifacts


In [ ]:
artifacts_root = Path("/kaggle/working")
if not artifacts_root.exists():
    artifacts_root = Path(".")

if "all_run_infos" not in globals() or not all_run_infos:
    raise RuntimeError("No run info found. Execute the run cell first.")

# Save combined summary tables
batch_summary_csv = artifacts_root / "batch_metrics_summary.csv"
batch_summary_json = artifacts_root / "batch_metrics_summary.json"

if "batch_summary_df" in globals() and isinstance(batch_summary_df, pd.DataFrame):
    batch_summary_df.to_csv(batch_summary_csv, index=False)
    print(f"Saved: {batch_summary_csv}")

    with batch_summary_json.open("w", encoding="utf-8") as f:
        json.dump(batch_summary_df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)
    print(f"Saved: {batch_summary_json}")


if "frame_quality_df_all" in globals() and isinstance(frame_quality_df_all, pd.DataFrame) and not frame_quality_df_all.empty:
    frame_quality_all_csv = artifacts_root / "frame_quality_all.csv"
    frame_quality_df_all.to_csv(frame_quality_all_csv, index=False)
    print(f"Saved: {frame_quality_all_csv}")

if "result_df" in globals() and isinstance(result_df, pd.DataFrame) and not result_df.empty:
    combined_debug_csv = artifacts_root / "debug_candidates_all.csv"
    result_df.to_csv(combined_debug_csv, index=False)
    print(f"Saved: {combined_debug_csv}")

zip_paths = []
for info in all_run_infos:
    out_dir = Path(info["output_dir"])
    if not out_dir.exists():
        continue

    src_debug_csv = Path(info["debug_csv_path"])
    if src_debug_csv.exists():
        target_name = f"{out_dir.name}_debug_candidates.csv"
        shutil.copy2(src_debug_csv, artifacts_root / target_name)


src_frame_quality_csv = Path(info.get("frame_quality_csv_path", ""))
if src_frame_quality_csv.exists():
    shutil.copy2(src_frame_quality_csv, artifacts_root / f"{out_dir.name}_frame_quality_debug.csv")

    src_metrics_csv = Path(info["metrics_csv_path"])
    src_metrics_json = Path(info["metrics_json_path"])
    if src_metrics_csv.exists():
        shutil.copy2(src_metrics_csv, artifacts_root / f"{out_dir.name}_metrics_summary.csv")
    if src_metrics_json.exists():
        shutil.copy2(src_metrics_json, artifacts_root / f"{out_dir.name}_metrics_summary.json")

    zip_base = artifacts_root / out_dir.name
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(out_dir))
    zip_paths.append(zip_path)

# Zip all per-case output directories together
if run_mode == "batch":
    batch_root = Path(BATCH_OUTPUT_ROOT)
    if batch_root.exists():
        batch_zip = shutil.make_archive(str(artifacts_root / "baseline_ocr_candidates_batch_all"), "zip", root_dir=str(batch_root))
        print(f"Batch zip created: {batch_zip}")

print("Per-case zip archives:")
for z in zip_paths:
    print("  -", z)
